In [1]:
from ase import Atoms
from ase.units import Ha
from ase_dirac import DIRAC
from pathlib import Path
import shutil


notebook_path=Path().parent.resolve()

h2_path = f'{notebook_path}/geometries/h2.xyz'
h2o_path = f'{notebook_path}/geometries/h2o.xyz'
ed_path = f'{notebook_path}/workdir/visual_tutorial_interactive/ed.cube'
elf_path = f'{notebook_path}/workdir/visual_tutorial_interactive/elf.cube'


---

# Dirac in python: Higher level handling of calculations
Up to this point, the `DIRAC` calculator was able to modify and run calculations, but we are going to build a cleaner interface. 

## Preparing and running calculations: 
The simplest way to perform a calculation is to provide a geometry, and chose one of the following options: 
1. Just input geometry: A calculator with default SCF behaviour will be used. 
2. Input geometry and a input file: A calculator based on the input file will be used.
3. Input geometry and a pass the inout as a string: A calculator based on the string will be used.
4. Input geometry and a template: A calculator based on the chosen template will be used.

In [2]:
DiracAtoms: Atoms = DIRAC.setup_geom_and_imp(h2_path)
DiracAtoms.get_potential_energy() / Ha

No input or template was selected. Building calculator with default SCF behaviour
Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule SCF.xyz with input SCF.inp
  creating archive file  SCF.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-1.1024686914212498

In [3]:
shutil.copyfile('SCF.inp', 'SCF2.inp')
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, inp_filename="SCF2.inp")
DiracAtoms.get_potential_energy() / Ha

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule SCF2.xyz with input SCF2.inp
  creating archive file  SCF2.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-1.1024686914212498

In [4]:
with open("SCF.inp") as f:
    inp_string = f.read() 
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, inp_string=inp_string)
DiracAtoms.get_potential_energy() / Ha

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule dirac.xyz with input dirac.inp
  creating archive file  dirac.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-1.1024686914212498

In [5]:
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, template_input_str="CCSD")
DiracAtoms.get_potential_energy() / Ha

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule CCSD.xyz with input CCSD.inp
  creating archive file  CCSD.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-1.144927725277986


---

## Modifying sections
But what if I don't like the content of the sections? They can be easily modified with the methods: 
- `DIRAC.override_section`: Overrides the content of a section with the inputed dirac-formated string.
- `DIRAC.remove_section`: Removes entirely a section. 
- `DIRAC.add_section`: Adds a section with the inputed dirac-formated string.

We will start with the previous example, we are building from the SCF template of H2, but we will now do the following: 
- Perform a DFT calculation. 
- Change occupation.
- Change the basis set. 

In [6]:
DiracAtomsMod = DIRAC.setup_geom_and_imp(h2_path, template_input_str="SCF")

print("Before modification:\n", DiracAtomsMod.calc.parameters)

DiracAtoms.calc.override_section("""**MOLECULE
*BASIS
.DEFAULT
cc-pvqz""")

DiracAtomsMod.calc.override_section("""
**HAMILTONIAN
.DFT
B3LYP
""")

DiracAtomsMod.calc.override_section("""
**WAVE FUNCTIONS
.SCF
*SCF
.CLOSED SHELL
2 2 
""")

print("\nAfter modification:\n", DiracAtomsMod.calc.parameters)

Before modification:
 {'dirac': {'.title': 'Relativistic SCF calculation', '.wave f': ''}, 'hamiltonian': {'.lvcorr': ''}, 'wave_function': {'.scf': ''}, 'molecule': {'*basis': {'.default': 'dyall.3zp\n'}}}

After modification:
 {'dirac': {'.title': 'Relativistic SCF calculation', '.wave f': ''}, 'hamiltonian': {'.DFT': 'B3LYP\n'}, 'wave_function': {'.SCF': '', '*SCF': {'.CLOSED SHELL': '2 2\n'}}, 'molecule': {'*basis': {'.default': 'dyall.3zp\n'}}}


In [7]:
DiracAtomsMod.get_potential_energy() / Ha

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule SCF.xyz with input SCF.inp
  creating archive file  SCF.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-0.8025772918219523

Or if we want to modify the wave function section to do CCSD too:

In [8]:
DiracAtomsMod = DIRAC.setup_geom_and_imp(h2_path, template_input_str="SCF")

print("Before modification:\n", DiracAtomsMod.calc.parameters)

DiracAtomsMod.calc.override_section("""
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    *SCF
    .PRINT 
    1
    **RELCCSD
    .ENERGY
    .PRINT
    1
""")


print("\nAfter modification:\n", DiracAtomsMod.calc.parameters, "\n")
DiracAtomsMod.get_potential_energy() / Ha

Before modification:
 {'dirac': {'.title': 'Relativistic SCF calculation', '.wave f': ''}, 'hamiltonian': {'.lvcorr': ''}, 'wave_function': {'.scf': ''}, 'molecule': {'*basis': {'.default': 'dyall.3zp\n'}}}

After modification:
 {'dirac': {'.title': 'Relativistic SCF calculation', '.wave f': ''}, 'hamiltonian': {'.lvcorr': ''}, 'wave_function': {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1\n'}}, 'molecule': {'*basis': {'.default': 'dyall.3zp\n'}}} 

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule SCF.xyz with input SCF.inp
  creating archive file  SCF.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-1.144927725277986


---


## After the calculation is over: retrieving the info
The results of the calculation are stored in the `DIRAC` object, and can be retrieved with the following methods:

In [9]:
DiracAtomsMod.calc.results.keys()

dict_keys(['SCF_energy', 'MP2_energy', 'CCSD_energy', 'energy', 'scf_energy', 'scf_eigenvalues'])

### h5 functionality from the calculator 
Under the hood what is happening is that the `read_results()` method in the calculator was extended to retrieve some data from the `CHECKPOINT.h5` file generated in the calculation:. We can see the content as:

In [10]:
DiracAtomsMod.calc.list_h5_contents[0:10]

['Group: input',
 'Group: input/aobasis',
 'Group: input/aobasis/1',
 'Dataset: input/aobasis/1/angular (1,)',
 'Dataset: input/aobasis/1/aobasis_id (1,)',
 'Dataset: input/aobasis/1/center (33,)',
 'Dataset: input/aobasis/1/contractions (11,)',
 'Dataset: input/aobasis/1/exponents (11,)',
 'Dataset: input/aobasis/1/n_ao (1,)',
 'Dataset: input/aobasis/1/n_cont (11,)']

And extract a property as:

In [11]:
n_ao = DiracAtomsMod.calc.extract_h5_value("input/aobasis/1/n_ao")
print(*n_ao)

30



---


# Visualization
It is possible to request visual properties. This is performed in two steps (here). First we set up and run the calculation as:

In [12]:
DiracAtomsVis = DIRAC.setup_geom_and_imp(h2o_path, template_input_str="SCF")
DiracAtomsVis.get_potential_energy() / Ha

Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent
 starting DIRAC for molecule SCF.xyz with input SCF.inp
  creating archive file  SCF.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-76.1185546456472

Then we can request the density as:

In [13]:
DiracAtomsVis.calc.request_visualization_density()

Or input directly the `.GRIDS` and `.GRIDFUCNTIONS`: 

In [14]:
DiracAtomsVis.calc.add_visualization_property(
    "id=2 input=create typ=1 npoints=[64,64,32] margin=2.0",
    "name=elf id_grid=2 purpose=visualization export=cube file_out=SCF_elf.cube",
)

DiracAtomsVis.calc.add_visualization_property(
    "id=3 input=dft",
    "name=ed id_grid=3 purpose=integration",
)

DiracAtomsVis.calc.parameters

{'dirac': {'.title': 'Relativistic SCF calculation', '.wave f': ''},
 'hamiltonian': {'.lvcorr': ''},
 'wave_function': {'.scf': ''},
 'molecule': {'*basis': {'.default': 'dyall.3zp\n'}}}

Then we compute the visual propertiees with:

In [15]:
DiracAtomsVis.calc.compute_visual_properties()

Since VISUAL keyword was detected in input, --get='*.cube' will be added to the run command.
Running Dirac with command pam --inp=PREFIX.inp --mol=PREFIX.xyz --silent --get="*.cube" --put="SCF.h5"
 starting DIRAC for molecule SCF.xyz with input SCF.inp
  creating archive file  SCF.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'SCF_elf.cube', 'SCF_ed.cube', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


And finally we can visualize the results interactively:

In [16]:
DiracAtomsVis.calc.plot_geom()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

See the list of availiable `.cube` files:

In [17]:
print(f'Availiable cube files (DiracAtomsVis.calc.list_cubes): {DiracAtomsVis.calc.list_cubes}' )

Availiable cube files (DiracAtomsVis.calc.list_cubes): ['SCF_elf.cube', 'SCF_ed.cube']


And plot them with the method:

In [18]:
DiracAtomsVis.calc.plot_property('SCF_ed.cube')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [19]:
view = DiracAtomsVis.calc.plot_property('SCF_elf.cube',  isovalue=0.9)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.